In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as sum_

In [2]:
spark = SparkSession.builder.appName("RDDvsDataFrame").getOrCreate()

# sample data: list of sales records (RDD)
sales_rdd = spark.sparkContext.parallelize([
    ("North",100),
    ("South",200),
    ("North",150),
    ("East", 50),
    ("South", 300)
])
print(sales_rdd)

26/09/06 08:53:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/06 08:53:11 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/06 08:53:11 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/09/06 08:53:11 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/09/06 08:53:11 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/09/06 08:53:11 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
26/09/06 08:53:11 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
26/09/06 08:53:11 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.
26/09/06 08:53:11 WARN Utils: Serv

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:274


In [5]:
#RDD approach: total sales or region
rdd_totals = (sales_rdd.map(lambda x:(x[0], x[1]))
              .reduceByKey(lambda a,b:a+b))

print("RDD Totals:", rdd_totals.collect())

RDD Totals: [('North', 250), ('East', 50), ('South', 500)]


In [9]:
#DataFrame approach
sales_df = spark.createDataFrame(sales_rdd, ["region", "amount"])
df_totals = (sales_df.groupBy("region").agg(sum_("amount").alias("total_sales")))
df_totals.show()

+------+-----------+
|region|total_sales|
+------+-----------+
| South|        500|
|  East|         50|
| North|        250|
+------+-----------+



In [12]:
spark2 = SparkSession.builder.appName("InMemoryDF").getOrCreate()

data = [
    (1, "Alice", 50000),
    (2, "Bob", 60000),
    (3, "Charlie", 55000)
]
df = spark2.createDataFrame(data,["id","name","salary"])
df.show()
df.printSchema()

+---+-------+------+
| id|   name|salary|
+---+-------+------+
|  1|  Alice| 50000|
|  2|    Bob| 60000|
|  3|Charlie| 55000|
+---+-------+------+

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)



In [18]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
spark3 = SparkSession.builder.appName("SchemaDF").getOrCreate()

data = [
    (1, "Alice", 50000),
    (2, "Bob", 60000),
    (3, "Charlie", 55000)
]

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(),True),
    StructField("salary", IntegerType(), False)
])
df = spark3.createDataFrame(data,schema)
df.show()
df.printSchema()

+---+-------+------+
| id|   name|salary|
+---+-------+------+
|  1|  Alice| 50000|
|  2|    Bob| 60000|
|  3|Charlie| 55000|
+---+-------+------+

root
 |-- id: integer (nullable = false)
 |-- name: string (nullable = true)
 |-- salary: integer (nullable = false)



In [19]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
spark4 = SparkSession.builder.appName("fromCsvDF").getOrCreate()

data = [
    (1, "Alice", 50000),
    (2, "Bob", 60000),
    (3, "Charlie", 55000)
]

schema = StructType([
    StructField("booth_number", IntegerType(), False),
    StructField("booth_price_level", StringType(),True),
    StructField("booth_description", StringType(), True),
    StructField("booth_type", StringType(), True)
])
df = spark4.read.option("header","true").schema(schema).csv("DataFarmersMarket/booth.csv")
df.show()
df.printSchema()

+------------+-----------------+--------------------+----------+
|booth_number|booth_price_level|   booth_description|booth_type|
+------------+-----------------+--------------------+----------+
|           1|                A|First booth to th...|  Standard|
|           2|                A|Second booth to t...|  Standard|
|           3|                B|Third booth to th...|     Small|
|           4|                C|First booth on le...|     Small|
|           5|                C|Second booth on l...|     Small|
|           6|                C|Third booth on le...|     Small|
|           7|                A|First booth to th...|  Standard|
|           8|                C|First booth on ri...|     Small|
|           9|                C|First booth on ri...|     Small|
|          10|                B|Back right booth ...|     Large|
|          11|                B|Back center booth...|     Large|
|          12|                B|Back left booth  ...|     Large|
+------------+-----------

In [ ]:
spark5 = SparkSession.builder.appName("TextFileDF").getOrCreate()

df_text = spar5.read.text("test.txt")
df_text.show()
df_text.printSchema()


In [ ]:
spark6 = SparkSesson.builder.appName("SQLDF").getOrCreate()

# need swql connection for this code to work fine
df = spark6.sql("""
    SELECT department, AVG(salary) as avg_salary
    FROM employees
    GROUP BY department
""")

df.printSchema()

In [ ]:
spark7 = SparkSession.builder.appName("JsonDF").getOrCreate()

df_json = spark.read.option("multiline","true").json("jsonfile.json")

df_json.show()
df_json.printSchema()

In [ ]:
spark8 = SparkSessoin.builder.appName("JdbcDF").getOrCreate()

jdbc_url = "jdbc:mysql://localhost:3306/employees_db"
table_name = "employee_details"
properties - {"user":"root", "password":"password","driver":"com.mysql.cj.jdbc.Driver"}

df_jdbc = spark8.read.jdbc(url=jdbc_url, table=table_name, properties=properties)

df_jdbc.show()
df_jdbc.printSchema()